# The CHSM8 tokenizer

Each move is one row of five integers, `(kind, src, dst, piece, promo)`, instead of an entry in a vocabulary of move
strings. The model predicts the four move fields with four small heads; a move's log-probability is their sum.
The tokenizer (`chsm8_tok`, Rust) reads any notation and checks legality.

In [ ]:
import os, sys
sys.path.insert(0, ".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
from chsm8 import tokenizer as T

Any notation, even mixed within one game:

In [ ]:
for game in ["1. e4 e5 2. Nf3 Nc6 3. Bb5", "e2e4 e7e5 g1f3 b8c6 f1b5", "e2-e4 e7-e5 Ng1-f3 Nb8-c6 Bf1-b5",
             "1. e4 e5 2. g1f3 Nc6 3. Bf1-b5"]:
    print(T.tokenize(game))

Tokens are `from | to << 6 | promo << 12`. `rows()` replays them to add the moving piece and the two legality
masks used in training (`occ`: squares of the side to move, `pmask`: piece types it still has).

In [ ]:
rows, occ, pmask = T.encode("1. e4 e5 2. Nf3 Nc6 3. Bb5")
names = "abcdefgh"
sq = lambda s: names[s % 8] + str(s // 8 + 1)
for (kind, src, dst, piece, promo), o, m in zip(rows, occ, pmask):
    if kind == T.KIND_MOVE:
        print(f"{sq(src)}{sq(dst)}  piece {piece}  promo {promo}  occ {o:016x}  pmask {m:06b}")
    else:
        print(["PAD", "BOS", "EOS"][kind])

Illegal moves are rejected:

In [ ]:
try:
    T.tokenize("1. e4 e5 2. Ke3")
except ValueError as e:
    print("ValueError:", e)

`analyze()` gives what the model needs to move: the rows so far and every legal next move.

In [ ]:
rows, legal = T.analyze("1. e4 e5 2. Nf3")
print(len(legal), "legal moves:", " ".join(san for _, san, _ in legal))